# 이미지 전처리: 클래스별 캡처를 640×640으로 변환

`dataset/클래스명/이미지` 구조를 그대로 유지하고, `dataset_640`에 결과를 저장합니다.
원본은 수정하지 않습니다. 큰 이미지는 비율을 유지해 축소하고 작은 이미지는 확대하지 않으며,
남는 공간은 흰 여백으로 채웁니다. PNG로 저장하고 기존 색상을 유지합니다.

이번 notebook에서는 해상도 변환만 수행합니다. 회전·그레이스케일 증강과 YOLO26x 학습은 다음 단계입니다.
위에서부터 셀을 순서대로 실행하세요. 경로 설정 셀의 `INPUT_DIR`을 실제 데이터 폴더에 맞추세요.


## 1. 라이브러리 설치
VS Code에서 Python 커널을 선택한 뒤 실행하세요. 이미 설치되어 있으면 그대로 유지됩니다.

In [ ]:
%pip install Pillow


## 2. 입력·출력 경로 설정

기본값은 현재 작업 폴더 안의 `dataset`입니다.
다른 위치라면 `INPUT_DIR = Path(r"D:\데이터\dataset")`처럼 수정하세요.
출력 폴더는 입력 폴더 옆의 `dataset_640`입니다.


In [ ]:
from pathlib import Path
import csv
from PIL import Image, ImageOps
from IPython.display import display

# 여기를 실제 dataset 경로에 맞추세요.
INPUT_DIR = Path.cwd() / "dataset"
OUTPUT_DIR = INPUT_DIR.with_name(INPUT_DIR.name + "_640")
SIZE = 640
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

print("현재 작업 폴더:", Path.cwd())
print("입력:", INPUT_DIR.resolve())
print("출력:", OUTPUT_DIR.resolve())


## 3. 변환 함수 정의

이 셀은 함수를 정의합니다. 아직 전체 데이터는 변환하지 않습니다.
변환 기록에는 원본 크기, 변환 후 크기, 배율, 여백 좌표를 저장합니다.
기존 YOLO `.txt` 라벨이 있으면 좌표 변환도 필요하므로 이 이미지 전용 코드는 중단합니다.


In [ ]:
def resize_image(path):
    """비율 유지 축소 + 중앙 배치. 작은 원본은 확대하지 않습니다."""
    with Image.open(path) as raw:
        image = ImageOps.exif_transpose(raw).convert("RGBA")
    original_width, original_height = image.size

    # 투명 배경은 흰색으로 합칩니다. 색상과 명암은 그대로 유지합니다.
    background = Image.new("RGBA", image.size, "white")
    background.alpha_composite(image)
    image = background.convert("RGB")
    image.thumbnail((SIZE, SIZE), Image.Resampling.LANCZOS)
    width, height = image.size
    left, top = (SIZE - width) // 2, (SIZE - height) // 2
    canvas = Image.new("RGB", (SIZE, SIZE), "white")
    canvas.paste(image, (left, top))

    transform = {
        "original_width": original_width, "original_height": original_height,
        "resized_width": width, "resized_height": height,
        "scale_x": width / original_width, "scale_y": height / original_height,
        "padding_left": left, "padding_top": top,
    }
    return canvas, transform


def main(input_dir=INPUT_DIR, output_dir=OUTPUT_DIR):
    input_dir, output_dir = Path(input_dir).resolve(), Path(output_dir).resolve()
    if not input_dir.is_dir():
        raise FileNotFoundError(f"INPUT_DIR을 실제 dataset 폴더로 수정하세요: {input_dir}")
    if output_dir == input_dir or output_dir.is_relative_to(input_dir):
        raise ValueError("결과 폴더는 원본 dataset 폴더 밖에 지정하세요.")
    if output_dir.exists():
        raise FileExistsError(f"덮어쓰지 않습니다. OUTPUT_DIR을 새 폴더로 바꾸세요: {output_dir}")

    files = sorted(p for p in input_dir.rglob("*")
                   if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    if not files:
        raise ValueError("이미지를 찾을 수 없습니다.")
    destinations = set()
    for path in files:
        relative = path.relative_to(input_dir)
        if len(relative.parts) < 2:
            raise ValueError(f"이미지를 클래스명 폴더 안에 넣으세요: {relative}")
        # 기존 정답 좌표를 잘못된 위치에 남기는 것을 방지합니다.
        if path.with_suffix(".txt").exists():
            raise ValueError(f"라벨 파일이 있습니다. 이미지와 박스 좌표를 함께 변환해야 합니다: {path}")
        destination = str(relative.with_suffix(".png")).casefold()
        if destination in destinations:
            raise ValueError(f"PNG 변환 후 파일명이 겹칩니다. 원본 파일명을 구분하세요: {relative}")
        destinations.add(destination)

    output_dir.mkdir(parents=True)
    report = output_dir / "resize_manifest.csv"
    with report.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = None
        for number, path in enumerate(files, 1):
            relative = path.relative_to(input_dir)
            try:
                image, transform = resize_image(path)
            except Exception as error:
                raise RuntimeError(f"이미지를 읽을 수 없습니다: {path}") from error
            destination = output_dir / relative.with_suffix(".png")
            destination.parent.mkdir(parents=True, exist_ok=True)
            image.save(destination, format="PNG")
            row = {"class_name": relative.parts[0], "source": str(relative),
                   "output": str(destination.relative_to(output_dir)), **transform}
            if writer is None:
                writer = csv.DictWriter(handle, fieldnames=list(row))
                writer.writeheader()
            writer.writerow(row)
            print(f"[{number}/{len(files)}] {relative} → 640×640", flush=True)
    print(f"완료: {len(files)}장\n저장 폴더: {output_dir}\n변환 기록: {report}")



## 4. 첫 이미지 미리보기
결과가 640×640인지, 이미지가 찌그러지거나 잘리지 않았는지 확인하세요. 파일은 저장하지 않습니다.

In [ ]:
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"2번 셀의 INPUT_DIR을 실제 dataset 폴더로 수정하세요: {INPUT_DIR}")
preview_files = sorted(p for p in INPUT_DIR.rglob("*")
                       if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
if not preview_files:
    raise ValueError("입력 폴더에서 이미지를 찾을 수 없습니다.")
preview_path = preview_files[0]
preview_image, preview_transform = resize_image(preview_path)
print("미리보기 파일:", preview_path.relative_to(INPUT_DIR))
print("원본 크기:", (preview_transform["original_width"], preview_transform["original_height"]))
print("내용 크기:", (preview_transform["resized_width"], preview_transform["resized_height"]))
print("저장될 이미지 크기:", preview_image.size)
display(preview_image)


## 5. 전체 이미지 변환 실행

이 셀을 실행하면 파일이 생성됩니다. 클래스별 폴더명과 하위 폴더 구조를 유지합니다.
이미 출력 폴더가 있으면 덮어쓰지 않습니다. 다시 실행할 때는 2번 셀에서 새 `OUTPUT_DIR`을 지정하세요.


In [ ]:
main(INPUT_DIR, OUTPUT_DIR)


## 6. 결과 크기와 클래스별 개수 확인

In [ ]:
with (OUTPUT_DIR / "resize_manifest.csv").open(encoding="utf-8-sig", newline="") as file:
    records = list(csv.DictReader(file))

class_counts = {}
for row in records:
    result_path = OUTPUT_DIR / row["output"]
    with Image.open(result_path) as result_image:
        assert result_image.size == (640, 640), result_path
        assert result_image.mode == "RGB", result_path
    class_counts[row["class_name"]] = class_counts.get(row["class_name"], 0) + 1

print(f"총 {len(records)}장: 전부 640×640 RGB PNG 확인 완료")
for class_name, count in sorted(class_counts.items()):
    print(f"{class_name}: {count}장")
print("결과 폴더:", OUTPUT_DIR.resolve())


## 다음 단계

결과 이미지를 확인한 뒤 회전·그레이스케일 증강을 진행합니다.
폴더명은 클래스 이름이며, 이번 단계에서 객체탐지용 정답박스를 자동으로 생성하지는 않습니다.
